# Stardew Valley Fishing AI — C51 DQN Training
Runs on Colab's free GPU (T4). Clone the repo, install deps, train the model, download the checkpoint.

In [ ]:
# @title 1. Clone the repository
import os
REPO_URL = "https://github.com/keethesh/StardewValleyFishingAI.git"  # @param {type:"string"}
BRANCH = "model-upgrade-v2"  # @param {type:"string"}

if not os.path.exists("StardewValleyFishingAI"):
    !git clone --branch {BRANCH} {REPO_URL}
else:
    print("Already cloned, pulling latest...")
    %cd StardewValleyFishingAI
    !git pull
    %cd ..

%cd StardewValleyFishingAI
print(f"Working in: {os.getcwd()}")

In [ ]:
# @title 2. Switch to the C51 model upgrade branch & install deps

# Force checkout the improvement branch
!git fetch origin
!git checkout -f {BRANCH}
!git pull origin {BRANCH}

# Install Python deps (no pygame needed for headless training)
!pip install torch numpy matplotlib 2>&1 | tail -3

In [ ]:
# @title 3. Verify GPU is active
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title 4. Install dummy pygame (headless) so rendering calls don't crash
import subprocess, sys

# Patch pygame to no-op so environment.py imports happily on Colab
pygame_patch = """
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'
"""
with open('colab_pygame_patch.py', 'w') as f:
    f.write(pygame_patch)

!pip install pygame 2>&1 | tail -1
print("Pygame installed (headless mode)")

In [ ]:
# @title 5. Patch environment.py for Colab's headless display
# Colab has no display, so rendering is disabled
!sed -i 's/render_mode="human"/render_mode=None/g' main.py
!sed -i 's/render_mode="human"  # Set to True/render_mode=None  # Set to True/g' main.py 2>/dev/null
print("Patched main.py for headless training")

# Make sure train_new_model is True
!sed -i 's/train_new_model = False/train_new_model = True/g' main.py
!sed -i 's/skip_evaluation = False/skip_evaluation = True/g' main.py
print("Training mode enabled")

In [ ]:
# @title 6. Start Training!

# Optional: override settings before training
NUM_EPISODES = 12000  # @param {type:"integer"}
SAVE_EVERY = 500     # @param {type:"integer"}

import os

# Create model directories
os.makedirs("models/checkpoints", exist_ok=True)
os.makedirs("training_logs/graphs", exist_ok=True)

print(f"Starting training: {NUM_EPISODES} episodes, save every {SAVE_EVERY}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("=" * 60)

# Run training
!python main.py

In [ ]:
# @title 7. Download trained model + logs
from google.colab import files
import glob
import os

# Find the latest checkpoint
checkpoints = sorted(glob.glob("models/checkpoints/*.pth"))
if checkpoints:
    latest = checkpoints[-1]
    print(f"Latest checkpoint: {latest}")
    files.download(latest)
else:
    print("No checkpoints found yet")

# Download milestone logs
logs = sorted(glob.glob("training_logs/milestones_*.txt"))
if logs:
    print(f"Milestone log: {logs[-1]}")
    files.download(logs[-1])

# Download training metrics CSV
csvs = sorted(glob.glob("training_logs/training_metrics_*.csv"))
if csvs:
    print(f"Metrics CSV: {csvs[-1]}")
    files.download(csvs[-1])

# Create a zip of all checkpoints
import zipfile
zip_path = "all_checkpoints.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    for ckpt in checkpoints:
        zf.write(ckpt, os.path.basename(ckpt))
print(f"Created {zip_path} ({len(checkpoints)} files)")

In [ ]:
# @title (Optional) Resume from a checkpoint
from google.colab import files
import os

# Upload your saved .pth file
uploaded = files.upload()

for fn in uploaded.keys():
    # Move to checkpoints folder
    os.makedirs("models/checkpoints", exist_ok=True)
    dest = os.path.join("models/checkpoints", fn)
    os.rename(fn, dest)
    print(f"Uploaded {fn} -> {dest}")
    
    # Update main.py to use fine-tune mode with this checkpoint
    with open('main.py', 'r') as f:
        content = f.read()
    content = content.replace('train_new_model = True', 'fine_tune_model = True')
    content = content.replace("model_path = 'models/YOUR_MODEL_NAME.pth'", f"model_path = '{dest}'")
    with open('main.py', 'w') as f:
        f.write(content)
    print(f"Set main.py to fine-tune from {dest}")
    print("Now run the Training cell again")